In [ ]:
import numpy as np
import cv2
import os
import csv
import matplotlib.pyplot as plt
import pandas as pd
import shutil


# split img


In [ ]:
def split_image(image, overlap_ratio, num_of_slice):
    height, width = image.shape[:2]
    
    part_width = width // 3
    part_height = height // 3
    
    overlap_x = int(part_width * overlap_ratio)
    overlap_y = int(part_height * overlap_ratio)
    
    parts = []
    positions = []
    positions_x_y = []

    
    for i in range(num_of_slice):
        for j in range(num_of_slice):
            left = max(0, j * part_width - overlap_x)
            upper = max(0, i * part_height - overlap_y)
            right = min(width, (j + 1) * part_width + overlap_x)
            lower = min(height, (i + 1) * part_height + overlap_y)

            t_l_x, t_l_y = left, upper
            t_r_x, t_r_y = right, upper
            b_l_x, b_l_y = left, lower
            b_r_x, b_r_y = right, lower
            
            
            
            part = image[upper:lower, left:right]
            parts.append(part)

            positions.append([left, upper, right, lower])
            positions_x_y.append([t_l_x, t_l_y, b_l_x, b_l_y, b_r_x, b_r_y, t_r_x, t_r_y])

            
    return parts, positions, positions_x_y

# show img

In [ ]:
def visualize_parts(image, parts, positions):
    for part, position in zip(parts, positions):
        print(position)
        plt.imshow(part)
        plt.axis('off')
        plt.show()

# check bbox


In [ ]:
def check_bbox_of_parts(parts, outer_bbox,inner_bbox):
    x1, y1, x2, y2 , x3, y3, x4, y4 = inner_bbox
    x1, y1, x2, y2 , x3, y3, x4, y4 = float(x1), float(y1), float(x2), float(y2), float(x3), float(y3), float(x4), float(y4)
    _x1, _y1, _x2, _y2 , _x3, _y3, _x4, _y4 = outer_bbox
    _x1, _y1, _x2, _y2 , _x3, _y3, _x4, _y4 = float(_x1), float(_y1), float(_x2), float(_y2), float(_x3), float(_y3), float(_x4), float(_y4)

    
    if ((_x1 <= x1 <= _x4) and (_y1 <= y1 <= _y2)) and ((_x1 <= x4 <= _x4) and (_y1 <= y4 <= _y2)) and \
    ((_x2 <= x2 <= _x3) and (_y1 <= y2 <= _y2)) and ((_x2 <= x3 <= _x3) and (_y4 <= y3 <= _y3)):
        return True
    return False

In [ ]:
def save_part(path, file_name, part):
    cv2.imwrite(os.path.join(path, file_name), part)

In [ ]:
def calculate_distance(position, bbox):
    x1, y1, x2, y2 , x3, y3, x4, y4 = bbox
    x1, y1, x2, y2 , x3, y3, x4, y4 = float(x1), float(y1), float(x2), float(y2), float(x3), float(y3), float(x4), float(y4)
    _x1, _y1, _x2, _y2 , _x3, _y3, _x4, _y4 = position
    _x1, _y1, _x2, _y2 , _x3, _y3, _x4, _y4 = float(_x1), float(_y1), float(_x2), float(_y2), float(_x3), float(_y3), float(_x4), float(_y4)

    _n_x1  = x1 - _x1
    _n_y1  = y1 - _y1
    _n_x2  = x2 - _x1
    _n_y2  = y2 - _y1
    _n_x3  = x3 - _x1
    _n_y3  = y3 - _y1
    _n_x4  = x4 - _x1
    _n_y4  = y4 - _y1
    
    _bbox = [_n_x1, _n_y1, _n_x2, _n_y2 , _n_x3, _n_y3, _n_x4, _n_y4]

    return _bbox

# Main

In [ ]:
save_path = r"D:\Work\codev2\code\train_data_2"
an_path = r"D:\Work\codev2\data\final_data\20.9 copy"
list_of_parts = []
for fol_name in os.listdir(an_path):
    folder_path = os.path.join(an_path, fol_name)
    for f_name in os.listdir(folder_path):
        file_path = os.path.join(folder_path, f_name)
        if f_name.endswith(".bmp")or f_name.endswith(".csv"):
            list_of_parts.append(file_path)

for path in list_of_parts:
    # path
    if not path.endswith(".bmp"):
        continue
    # path = r"D:\Work\codev2\code\work_code\model_code\090.bmp"
    path_csv = path[:-4]+".csv"
    # save_path = r"D:\Work\codev2\code\work_code\model_code"
    ##########################################################
    #############################################

    # image split
    image = cv2.imread(path)
    parts, positions, positions_x_y = split_image(image, 0.1, 3)

    for i ,part in enumerate(parts):
        print(i)
        save_part(save_path, path[-7:-4]+f"_{i}.bmp", part)
    # visualize_parts(image, parts, positions)
    #########################################################
    # csv split
    part_1 = []
    part_2 = []
    part_3 = []
    part_4 = []
    part_5 = []
    path_6 = []
    path_7 = []
    path_8 = []
    path_9 = []

    with open(path_csv, 'r') as csvfile:
        reader = csv.reader(csvfile)
        for i ,row in enumerate(reader):
            if i == 0:
                
                continue
            bbox = row[1:]
            # check bbox of each part
            for i,position in enumerate(positions_x_y):
                # print(i, position)
                # print(position)
                if check_bbox_of_parts(parts,position, bbox) == True:
                    # print(position )
                    if i == 0:
                        _bbox = calculate_distance(position, bbox)
                        part_1.append([_bbox, position])
                    if i == 1:
                        _bbox = calculate_distance(position, bbox)
                        part_2.append([_bbox, position])
                    if i == 2:
                        _bbox = calculate_distance(position, bbox)
                        part_3.append([_bbox, position])
                    if i == 3:
                        _bbox = calculate_distance(position, bbox)
                        part_4.append([_bbox, position])
                    if i == 4:
                        _bbox = calculate_distance(position, bbox)
                        part_5.append([_bbox, position])
                    if i == 5:
                        _bbox = calculate_distance(position, bbox)
                        path_6.append([_bbox, position])
                    if i == 6:
                        _bbox = calculate_distance(position, bbox)
                        path_7.append([_bbox, position])
                    if i == 7:
                        _bbox = calculate_distance(position, bbox)
                        path_8.append([_bbox, position])
                    if i == 8:
                        _bbox = calculate_distance(position, bbox)
                        path_9.append([_bbox, position])

    part_1,part_2,part_3,part_4,part_5,path_6,path_7,path_8,path_9 = pd.DataFrame(part_1),pd.DataFrame(part_2),pd.DataFrame(part_3),pd.DataFrame(part_4),\
        pd.DataFrame(part_5),pd.DataFrame(path_6),pd.DataFrame(path_7),pd.DataFrame(path_8),pd.DataFrame(path_9)
        

    part_1.to_csv(os.path.join(save_path,path[-7:-4]+f"_{0}.csv"))
    part_2.to_csv(os.path.join(save_path,path[-7:-4]+f"_{1}.csv"))
    part_3.to_csv(os.path.join(save_path,path[-7:-4]+f"_{2}.csv"))
    part_4.to_csv(os.path.join(save_path,path[-7:-4]+f"_{3}.csv"))
    part_5.to_csv(os.path.join(save_path,path[-7:-4]+f"_{4}.csv"))
    path_6.to_csv(os.path.join(save_path,path[-7:-4]+f"_{5}.csv"))
    path_7.to_csv(os.path.join(save_path,path[-7:-4]+f"_{6}.csv"))
    path_8.to_csv(os.path.join(save_path,path[-7:-4]+f"_{7}.csv"))
    path_9.to_csv(os.path.join(save_path,path[-7:-4]+f"_{8}.csv"))

                # break
            # break
                    

# csv to json 

In [ ]:
import os
import json
import csv
import shutil
import pandas as pd
import ast
import json

class Load_Data:
    def __init__(self, data_path, new_path):
        self.data_path  = data_path
        self.new_path = new_path

    # def load_data(self):
    #     for folder in os.listdir(self.data_path):
    #         print(folder)
    #         os.makedirs(os.path.join(self.new_path, folder), exist_ok=True)
    #         for file in os.listdir(os.path.join(self.data_path, folder)):
    #             if file.endswith(".csv"):
    #                 shutil.copy(os.path.join(self.data_path, folder, file), os.path.join(self.new_path, folder, file))
    #             if file.endswith(".bmp"):
    #                 shutil.copy(os.path.join(self.data_path, folder, file), os.path.join(self.new_path, folder, file))
    #             if file.endswith(".json"):
    #                 os.remove(os.path.join(self.new_path, folder, file))    
    def make_labelme_template(self):
       
            for file in os.listdir(os.path.join(self.new_path)):
                label = ""
                list_of_name = ["053", "019", "066", "067", "068", "259", "260", "198", "261", "134", "157"]
                # print(file[:3])
                # break
                if file[:3] in list_of_name:
                    label = "black_box"
                else:
                    label = "numb_box"
                if file.endswith(".csv"):
                    with open(os.path.join(self.new_path, file), 'r') as f:
                        csv_reader = pd.read_csv(f)
                        shapes = []
                        for index, row in csv_reader.iterrows():
                            row = row[1]
                            row = json.loads(row)
                            # print(type(row))
                            # break
                            shape = {
                                "label": label,
                                "points": [
                                    [row[0], row[1]],
                                    [row[2], row[3]],
                                    [row[4], row[5]],
                                    [row[6], row[7]],
                                ],
                            "group_id": None,
                            "description": "",
                            "shape_type": "polygon",
                            "flags": {},
                            "mask": None          
                            }
                            shapes.append(shape)

                        data = {
                            "version": "5.5.0",
                            "flags": {},
                            "shapes": shapes,
                            "imagePath": file[:-4] + ".bmp",
                            "imageData": None,
                            "imageHeight": 1337 ,
                            "imageWidth": 2188 
                        }

                        with open(os.path.join(self.new_path, file[:-4] + ".json"), 'w') as f:
                            json.dump(data, f, indent=4) 
    
                            
def main():
    path = r"D:\Work\codev2\code\train_data_2/"
    new_path = r"D:\Work\codev2\code\train_data_2"
    data = Load_Data(path, new_path)
    # data.load_data()
    data.make_labelme_template()
main()

# predict


In [ ]:
import os


prediction_path =  r"D:\Work\codev2\code\work_code\model_code\runs\detect\predict25\labels"

test_data = r"D:\Work\codev2\code\test_data"


for file in os.listdir(test_data):
    print(file)